# Verbessertes Modell 1 – Financial Inclusion in Africa
##### Ziel: Baseline nachbauen (F1 0,49) und mit Feature Engineering verbessern.

In [ ]:
import pandas as pd  # Werkzeugkasten für Tabellen  # Kap. 15
from sklearn.compose import ColumnTransformer  # verschiedene Spalten unterschiedlich behandeln  # neu
from sklearn.pipeline import Pipeline  # alle Schritte zu einem Fließband verbinden  # neu
from sklearn.preprocessing import OneHotEncoder  # Text-Spalten in 0/1-Spalten umwandeln  # neu
from sklearn.preprocessing import StandardScaler  # Zahlen-Spalten auf gleiche Skala bringen  # neu
from sklearn.linear_model import LogisticRegression  # das Modell  # neu
from sklearn.model_selection import StratifiedKFold  # 5 faire Teile für die Kreuzvalidierung  # neu
from sklearn.model_selection import cross_val_score  # Kreuzvalidierung ausführen  # neu

In [ ]:
df = pd.read_csv("data/raw/Train.csv")  # Daten laden vom Hauptordner aus  # Kap. 15
df["target"] = (df["bank_account"] == "Yes").astype(int)  # Yes wird 1, No wird 0  # neu
print(df.shape)  # Kontrolle: Zeilen und Spalten  # Kap. 15

In [ ]:
kategorien = [  # Text-Spalten, die per One-Hot umgewandelt werden  # Kap. 6
    "country",  # Land  # Kap. 6
    "location_type",  # Stadt oder Land  # Kap. 6
    "cellphone_access",  # Handyzugang  # Kap. 6
    "gender_of_respondent",  # Geschlecht  # Kap. 6
    "relationship_with_head",  # Beziehung zum Haushaltsvorstand  # Kap. 6
    "marital_status",  # Familienstand  # Kap. 6
    "education_level",  # Bildung  # Kap. 6
    "job_type",  # Art der Arbeit  # Kap. 6
]  # Ende der Liste  # Kap. 6
zahlen = ["household_size", "age_of_respondent"]  # Zahlen-Spalten, die standardisiert werden  # Kap. 6
X = df[kategorien + zahlen]  # Merkmale: alle außer uniqueid, year, bank_account, target  # Kap. 15
y = df["target"]  # Ziel  # Kap. 15
print(X.shape)  # Kontrolle  # Kap. 15

In [ ]:
vorbereitung = ColumnTransformer(  # Spalten je nach Typ unterschiedlich vorbereiten  # neu
    transformers=[  # Liste der Vorbereitungs-Schritte  # neu
        ("text", OneHotEncoder(handle_unknown="ignore"), kategorien),  # Text-Spalten in 0/1 umwandeln  # neu
        ("zahl", StandardScaler(), zahlen),  # Zahlen-Spalten auf gleiche Skala bringen  # neu
    ]  # Ende der Liste  # neu
)  # Ende der Vorbereitung  # neu

modell = LogisticRegression(class_weight="balanced", max_iter=1000)  # Modell, das Ungleichgewicht ausgleicht  # neu

pipeline = Pipeline(  # das Fließband  # neu
    steps=[  # Reihenfolge der Stationen  # neu
        ("vorbereitung", vorbereitung),  # Station 1: Daten vorbereiten  # neu
        ("modell", modell),  # Station 2: Modell lernt und sagt vorher  # neu
    ]  # Ende der Stationen  # neu
)  # Ende des Fließbands  # neu
pipeline  # Fließband anzeigen  # neu

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 5 Teile, gemischt, fairer Anteil Yes  # neu
f1_werte = cross_val_score(pipeline, X, y, cv=cv, scoring="f1")  # 5-mal trainieren und F1 messen  # neu
print("F1 pro Durchgang:", f1_werte.round(3))  # die 5 einzelnen Ergebnisse  # Kap. 14
print("F1 Durchschnitt:", f1_werte.mean().round(3))  # der Durchschnitt = unser Ergebnis  # Kap. 14

In [ ]:
df["handy_x_bildung"] = df["cellphone_access"] + "_" + df["education_level"]  # Kombination Handy und Bildung  # neu
df["bildung_x_land"] = df["education_level"] + "_" + df["country"]  # Kombination Bildung und Land  # neu
df["ort_x_land"] = df["location_type"] + "_" + df["country"]  # Kombination Stadt/Land und Land  # neu
print(df["handy_x_bildung"].value_counts().head())  # Kontrolle: die häufigsten Kombinationen  # Kap. 15

In [ ]:
kategorien_neu = kategorien + ["handy_x_bildung", "bildung_x_land", "ort_x_land"]  # alte plus neue Text-Spalten  # Kap. 6
X_neu = df[kategorien_neu + zahlen]  # neue Merkmalstabelle  # Kap. 15

vorbereitung_neu = ColumnTransformer(  # Weiche mit den neuen Spalten  # neu
    transformers=[  # Liste der Vorbereitungs-Schritte  # neu
        ("text", OneHotEncoder(handle_unknown="ignore"), kategorien_neu),  # Text-Spalten in 0/1  # neu
        ("zahl", StandardScaler(), zahlen),  # Zahlen-Spalten skalieren  # neu
    ]  # Ende der Liste  # neu
)  # Ende der Vorbereitung  # neu

pipeline_neu = Pipeline(  # neues Fließband  # neu
    steps=[  # Stationen  # neu
        ("vorbereitung", vorbereitung_neu),  # Station 1  # neu
        ("modell", LogisticRegression(class_weight="balanced", max_iter=1000)),  # Station 2  # neu
    ]  # Ende der Stationen  # neu
)  # Ende des Fließbands  # neu

f1_neu = cross_val_score(pipeline_neu, X_neu, y, cv=cv, scoring="f1")  # 5-mal trainieren und messen  # neu
print("F1 pro Durchgang:", f1_neu.round(3))  # Einzelwerte  # Kap. 14
print("F1 Durchschnitt:", f1_neu.mean().round(3))  # Vergleich mit 0,494  # Kap. 14

In [ ]:
from sklearn.model_selection import GridSearchCV  # mehrere Einstellungen automatisch ausprobieren  # neu

einstellungen = {"modell__C": [0.01, 0.1, 1, 10, 100]}  # Werte für den Knopf C  # neu

suche = GridSearchCV(  # Suche nach der besten Einstellung  # neu
    pipeline_neu,  # unser Fließband  # neu
    param_grid=einstellungen,  # die Werte, die probiert werden  # neu
    cv=cv,  # dieselben 5 Teile wie vorher  # neu
    scoring="f1",  # bewertet nach F1  # neu
)  # Ende der Suche  # neu
suche.fit(X_neu, y)  # Suche starten: 5 Werte × 5 Durchgänge = 25 Trainings  # neu
print("Bestes C:", suche.best_params_)  # welche Einstellung gewonnen hat  # neu
print("Bester F1:", round(suche.best_score_, 3))  # F1 mit der besten Einstellung  # Kap. 14

In [ ]:
from sklearn.model_selection import cross_val_predict  # Vorhersagen aus der Kreuzvalidierung holen  # neu
from sklearn.metrics import f1_score  # F1-Score berechnen  # neu

bestes_modell = suche.best_estimator_  # das Fließband mit dem besten C aus Schritt 7  # neu
wahrsch = cross_val_predict(bestes_modell, X_neu, y, cv=cv, method="predict_proba")  # Wahrscheinlichkeiten für jede Person  # neu
wahrsch_ja = wahrsch[:, 1]  # nur die Spalte „Wahrscheinlichkeit für Ja“  # Kap. 14
print(wahrsch_ja[:10].round(2))  # die ersten 10 Wahrscheinlichkeiten ansehen  # Kap. 14

In [ ]:
schwellen = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]  # Grenzen, die wir testen  # Kap. 6
for s in schwellen:  # jede Grenze nacheinander  # Kap. 9
    vorhersage = (wahrsch_ja >= s).astype(int)  # ab dieser Grenze „Ja“ (1), sonst „Nein“ (0)  # neu
    print(s, round(f1_score(y, vorhersage), 3))  # Grenze und F1-Score ausgeben  # Kap. 9

In [ ]:
from sklearn.metrics import precision_score  # Precision berechnen  # neu
from sklearn.metrics import recall_score  # Recall berechnen  # neu
from sklearn.metrics import mean_absolute_error  # MAE berechnen (Zindi-Metrik)  # neu

beste_schwelle = 0.5  # Startwert  # Kap. 4
bester_f1 = 0  # Startwert  # Kap. 4
for s in schwellen:  # alle Grenzen durchgehen  # Kap. 9
    f1 = f1_score(y, (wahrsch_ja >= s).astype(int))  # F1 bei dieser Grenze  # neu
    if f1 > bester_f1:  # besser als bisher?  # Kap. 8
        bester_f1 = f1  # neuen Bestwert merken  # Kap. 4
        beste_schwelle = s  # passende Grenze merken  # Kap. 4

vorhersage_best = (wahrsch_ja >= beste_schwelle).astype(int)  # Vorhersage mit der besten Grenze  # neu

print("Baseline F1 (Schritt 4):", f1_werte.mean().round(3))  # Ausgangswert  # Kap. 14
print("Mit Interaktionen F1 (Schritt 6):", f1_neu.mean().round(3))  # nach Feature Engineering  # Kap. 14
print("Nach Tuning F1 (Schritt 7):", round(suche.best_score_, 3), suche.best_params_)  # nach Tuning  # Kap. 14
print("Beste Schwelle:", beste_schwelle)  # gewählte Grenze  # Kap. 3
print("F1 (Yes):", round(f1_score(y, vorhersage_best), 3))  # Endergebnis Modell 1  # neu
print("Precision (Yes):", round(precision_score(y, vorhersage_best), 3))  # Trefferquote der Ja-Meldungen  # neu
print("Recall (Yes):", round(recall_score(y, vorhersage_best), 3))  # gefundene Kontoinhaber  # neu
print("MAE (Zindi):", round(mean_absolute_error(y, vorhersage_best), 3))  # Anteil falscher Vorhersagen  # neu

## Fazit: Verbessertes Modell 1

| Stufe | F1 (Yes) |
| --- | --- |
| Baseline (Logistische Regression, balanced, alle Merkmale) | 0,494 |
| + Interaktionen (Handy × Bildung, Bildung × Land, Stadt/Land × Land) | 0,502 |
| + Tuning (bestes C = 0,1) | 0,502 |
| **+ Schwelle 0,7 statt 0,5** | **0,550** |

**Endergebnis Modell 1:** F1 = 0,55 · Precision = 0,535 · Recall = 0,566 (5-fache CV)

**Einordnung:** Die Interaktionen bringen ein kleines Plus, das Tuning von C keines. Den größten Effekt hat die Schwelle: Erst ab 70 % Wahrscheinlichkeit „Ja“ zu sagen, reduziert Fehlalarme. Precision und Recall sind danach ausgeglichen. Das Projektziel (F1 ≥ 0,55) ist erreicht.

**Einschränkung:** Die Schwelle wurde auf denselben CV-Vorhersagen gewählt, auf denen gemessen wurde. Das Ergebnis ist daher leicht optimistisch und wird in Milestone 3 auf einem separaten Testteil geprüft.